# Kaggle runner — LLM reliability study (MMLU-Pro)
Settings: **GPU T4 x2** (or P100), Internet **on**. Add your Hugging Face token as a Kaggle secret named `HF_TOKEN` (needed for gated Llama).

Run cells top to bottom. Generation is resumable: if the session dies, re-run from the generation cell.

In [ ]:
# 1. Get the code (upload the repo as a Kaggle dataset, or clone your GitHub copy)
!git clone https://github.com/krisy1/llm-reliability.git
%cd llm-reliability
!pip install -q -r requirements.txt

In [ ]:
# 2. Optional: vLLM is 5-10x faster if it installs on the current Kaggle image
!pip install -q vllm || echo 'vLLM not available -> use the transformers backend'

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
# 3. Data, reshuffles, manifest (wordings must already be finalized; copy data/reworded/ into the repo)
!python -m src.sample_mmlu --n 300
!python -m src.make_variants orders --items data/sampled/mmlu_pro_n300.jsonl
!python -m src.build_manifest --items data/sampled/mmlu_pro_n300.jsonl --seeds 0 1 2 3 --wordings w0 w1 w2 --name mmlu300_full

In [ ]:
# 4. Time the largest model on 50 items first (M1 milestone)
!python -m src.build_manifest --items data/sampled/mmlu_pro_n300.jsonl --seeds 0 1 2 3 --wordings w0 w1 w2 --limit 50 --name timing50
!python -m src.run_generation --model phi3.5-mini --manifest data/manifests/timing50.jsonl --batch-size 32
!tail -1 results/logs/timing.jsonl

In [ ]:
# 5. Full run, one model per cell (resumable)
MODEL = 'qwen2.5-3b'   # then llama3.2-3b, phi3.5-mini, smollm3-3b, granite3.3-2b
!python -m src.run_generation --model {MODEL} --manifest data/manifests/mmlu300_full.jsonl --batch-size 32
# with vLLM: add --backend vllm

In [ ]:
# 6. Score + analyse
!python -m src.score_runs --manifest data/manifests/mmlu300_full.jsonl
!python -m src.analyse --scored results/scored/mmlu300_full.csv --ranking-only

## BFCL multiple (function calling)
Same steps with `--config configs/experiment_bfcl.yaml`. Rewordings: run `make_variants reword` on `data/sampled/bfcl_multiple_n200.jsonl` locally first (checks that every argument value survives verbatim), review, `finalize`, and copy `data/reworded/` here.

In [ ]:
!python -m src.sample_bfcl
!python -m src.make_variants orders --items data/sampled/bfcl_multiple_n200.jsonl --config configs/experiment_bfcl.yaml
!python -m src.build_manifest --config configs/experiment_bfcl.yaml --items data/sampled/bfcl_multiple_n200.jsonl --name bfcl200_full

In [ ]:
MODEL = 'qwen2.5-3b'
!python -m src.run_generation --config configs/experiment_bfcl.yaml --model {MODEL} --manifest data/manifests/bfcl200_full.jsonl --batch-size 16
!python -m src.score_runs --manifest data/manifests/bfcl200_full.jsonl
!python -m src.analyse --config configs/experiment_bfcl.yaml --scored results/scored/bfcl200_full.csv --ranking-only

In [ ]:
# 7. Save outputs before the session ends
!zip -qr /kaggle/working/results.zip results data/manifests